In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from PIL import Image

from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification

**Setup and model load**

In [3]:
MODEL_DIR = "/content/drive/MyDrive/Fine tuned models/swin_beans_best"
OUTPUT_DIR = "/content/drive/MyDrive/Fine tuned models/swin_beans_best/images/multi_scale_inspection.png"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = AutoImageProcessor.from_pretrained(MODEL_DIR)

model = AutoModelForImageClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

Loading weights:   0%|          | 0/221 [00:00<?, ?it/s]

SwinForImageClassification(
  (swin): SwinModel(
    (embeddings): SwinEmbeddings(
      (patch_embeddings): SwinPatchEmbeddings(
        (projection): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): SwinEncoder(
      (layers): ModuleList(
        (0): SwinStage(
          (blocks): ModuleList(
            (0): SwinLayer(
              (attention): SwinAttention(
                (q_proj): Linear(in_features=96, out_features=96, bias=True)
                (k_proj): Linear(in_features=96, out_features=96, bias=True)
                (v_proj): Linear(in_features=96, out_features=96, bias=True)
                (o_proj): Linear(in_features=96, out_features=96, bias=True)
                (relative_position_bias): SwinRelativePositionBias()
              )
              (layernorm_before): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
         

In [4]:
dataset = load_dataset("AI-Lab-Makerere/beans")

print("Device :", device)
print("Classes :", model.config.id2label)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  144MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.5MB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1034 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/133 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/128 [00:00<?, ? examples/s]

Device : cuda
Classes : {0: 'angular_leaf_spot', 1: 'bean_rust', 2: 'healthy'}


#### Extract prediction and stage features

In [5]:
def extract_swin_features(image):
    inputs = processor(
        images=image.convert("RGB"),
        return_tensors="pt"
    )
    pixel_values = inputs["pixel_values"].to(device)
    
    with torch.inference_mode():
        backbone_outpus = model.swin(
            pixel_values=pixel_values,
            output_hidden_states=True,
            output_hidden_states_before_downsampling=True,
            return_dict=True
        )

        logits = model.classifier(backbone_outpus.pooler_output)
    
    probabilities = torch.softmax(logits, dim=-1)[0]
    
    stage_features = list(backbone_outpus.reshaped_hidden_states[-4:])
    return (
        pixel_values.cpu(),
        probabilities.cpu(),
        [
            feature.cpu() for feature in stage_features
        ],
    )